# Does the projection head explain MobileNetV2?

`encoder_head` is `Linear(d, 128)`: never pruned, discarded at eval, and
carrying 69.7% of the first-order contrastive descent. As a fraction of the
weights that actually survive pruning:

| | 0.95 | 0.97 | 0.99 | 0.999 |
|---|---:|---:|---:|---:|
| ResNet-34 (d=512, 21.3M prunable) | 6% | 10% | 31% | 309% |
| **MobileNetV2** (d=1280, 2.2M prunable) | **149%** | **248%** | **744%** | 7444% |

MobileNetV2's $\Delta$ is negative at every sparsity and gets worse as sparsity
rises — −1.77, −2.51, −4.80 under magnitude — which is the signature of an
objective spending its gradient on a free adapter bigger than the network it is
meant to preserve.

Six arms, MobileNetV2 / magnitude, at 0.95 (ratio 149%) and 0.99 (ratio 744%):

| arm | head | ratio at 0.99 | what it changes |
|---|---|---:|---|
| `head128` | Linear(1280,128), trainable | 744% | the grid's setting, control |
| `head64` | Linear(1280,64) | 372% | capacity **and** geometry |
| `head32` | Linear(1280,32) | 186% | " |
| `head16` | Linear(1280,16) | 93% | " |
| `headfrozen` | Linear(1280,128), frozen+shared | 0% trainable | capacity only |
| `headnone` | no head, contrast pooled features | 0% | capacity only |

The width ladder and the frozen/none arms answer different questions. If only
the narrow heads help, the contrastive space was too wide. If frozen and none
help too, it is the trainable adapter, and width is incidental.

ResNet-34 at 0.999 runs `head128` against `head16` as the opposite control:
BaCP wins there, so a shrink that helps MobileNetV2 and hurts ResNet-34 would
show the head is doing real work where there is backbone left to work with.

One seed. This locates a mechanism; it does not publish a number.


In [ ]:
import sys, pathlib

here = pathlib.Path.cwd()
while not (here / '.git').exists() and here != here.parent:
    here = here.parent
sys.path.insert(0, str(here / 'project' / 'test_notebooks'))

import nb_common as nb
info = nb.setup()


## Plan

In [ ]:
SEED   = 1
GPU    = 0
PRUNER = 'magnitude'

# (tag, kwargs) -- everything else stays at the grid's BaCP settings
ARMS = [
    ('head128',    dict(num_out_features=128)),                      # control
    ('head64',     dict(num_out_features=64)),
    ('head32',     dict(num_out_features=32)),
    ('head16',     dict(num_out_features=16)),
    ('headfrozen', dict(num_out_features=128, proj_mode='tied_frozen')),
    ('headnone',   dict(num_out_features=128, proj_mode='none')),
]

plan = []
for tag, kw in ARMS:
    for sp in (0.95, 0.99):
        plan.append(nb.make_cell('mobilenet_v2', 'bacp', seed=SEED, pruner=PRUNER,
                                 sparsity=sp, variant=tag, **kw))
# opposite control: ResNet-34 at 0.999, where BaCP wins and the ratio is 309%
for tag, kw in (('head128', dict(num_out_features=128)),
                ('head16', dict(num_out_features=16))):
    plan.append(nb.make_cell('resnet34', 'bacp', seed=SEED, pruner=PRUNER,
                             sparsity=0.999, variant=tag, **kw))

# only the head may differ from the grid's BaCP arm
for c in plan:
    ref = nb.FAMILIES[c['model_name']]['bacp']
    for k in ('learning_rate', 'epochs', 'epochs_ft', 'delta_T', 'sparsity_scheduler',
              'recovery_epochs', 'val_split', 'prune_task_head', 'wanda_group',
              'optimizer_type', 'batch_size', 'num_classes', 'dataset_name',
              'tau', 'contrastive_mode', 'lambdas', 'n_views'):
        if k in ref:
            assert c['config'][k] == ref[k], (c['key'], k, c['config'][k], ref[k])
    assert c['config']['epochs'] == 50 and c['config']['epochs_ft'] == 25

DIM = {'mobilenet_v2': 1280, 'resnet34': 512}
PRUNABLE = {'mobilenet_v2': 2202560, 'resnet34': 21265088}
print('%d runs' % len(plan))
print()
print('%-12s %-13s %-7s %10s %10s %8s' %
      ('arm', 'model', 'spars', 'head', 'surviving', 'ratio'))
print('-' * 68)
for c in plan:
    m = c['model_name']; tag = c['key'].rsplit('.', 1)[-1]
    sp = float(c['config']['target_sparsity'])
    trainable = 0 if c['config'].get('proj_mode') in ('tied_frozen', 'none') \
        else DIM[m] * c['config']['num_out_features'] + c['config']['num_out_features']
    surv = PRUNABLE[m] * (1 - sp)
    print('%-12s %-13s %-7s %10s %10s %7.0f%%'
          % (tag, m, sp, '{:,}'.format(trainable), '{:,.0f}'.format(surv),
             100 * trainable / surv))
print()
print('est ~%.0f min' % sum(9.9 if c['model_name'] == 'mobilenet_v2' else 8.1
                            for c in plan))
assert nb.sanity_check(plan), 'sanity check failed'


## Run

In [ ]:
nb.run_group(plan, gpu=GPU)


## Verdict

In [ ]:
import json, glob, os, statistics as st

root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    try:
        r = json.load(open(f, encoding='utf-8'))
    except Exception:
        continue
    if r.get('status') == 'ok':
        acc[r.get('experiment_group') or ''] = (
            r.get('test_acc_exact_pct') or r.get('test_acc_pct'))

def show(model, sp, ip_ref):
    print()
    print('%s / magnitude / %s   (I.P. baseline %s)' % (model, sp, ip_ref))
    print('  %-12s %9s %9s' % ('arm', 'acc', 'vs head128'))
    base = acc.get('static.bacp.%s.cifar10.s%s.magnitude.seed%d.head128'
                   % (model, sp, SEED))
    for tag, _kw in ARMS:
        v = acc.get('static.bacp.%s.cifar10.s%s.magnitude.seed%d.%s'
                    % (model, sp, SEED, tag))
        if v is None:
            continue
        d = '' if base is None or tag == 'head128' else '%+9.2f' % (v - base)
        print('  %-12s %9.2f %9s' % (tag, v, d))

show('mobilenet_v2', 0.95, '89.72')
show('mobilenet_v2', 0.99, '85.61')
print()
print('resnet34 / magnitude / 0.999   (I.P. baseline 76.32, BaCP 81.65)')
for tag in ('head128', 'head16'):
    v = acc.get('static.bacp.resnet34.cifar10.s0.999.magnitude.seed%d.%s' % (SEED, tag))
    if v is not None:
        print('  %-12s %9.2f' % (tag, v))
print()
print('If the narrow heads recover MobileNetV2 and head16 costs ResNet-34, the')
print('head is a capacity sink whose harm scales with how little backbone is left.')
